<a href="https://colab.research.google.com/github/y2d-kr/colab_ngrok_ollama_langchain/blob/main/colab_ollama_ngrok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Update apt-get and install zstd, which is a dependency for Ollama.
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

# Run the official Ollama installation script.
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os

# Set the OLLAMA_HOST environment variable to allow external access within Colab.
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

# Start the Ollama server in the background.
# It's important to set OLLAMA_HOST again for the subprocess.
print("Starting Ollama server...")
ollama_process = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"})
time.sleep(5) # Give the server a moment to start up
print("Ollama server started.")

# Pull a model, e.g., 'mistral', to get started.
print("Pulling mistral model...")
# !ollama pull mistral
!ollama pull qwen3.5
print("Qwen3.5 model pulled. Ollama is ready to use!")

### ngrok 설정

1.  **ngrok 웹사이트에서 인증 토큰 가져오기**: ngrok 계정이 없으면 [ngrok 웹사이트](https://ngrok.com/)에서 가입하고 `Your Authtoken` 페이지에서 인증 토큰을 받으세요.
2.  **Colab 비밀 저장소에 토큰 저장**: Colab 왼쪽 패널의 '🔑' 아이콘을 클릭하여 '비밀' 탭을 엽니다. `NGROK_AUTH_TOKEN`이라는 새 비밀을 추가하고 ngrok 인증 토큰 값을 입력하세요.

In [ ]:
# ngrok 설치
!pip install pyngrok -qq

from pyngrok import ngrok
from google.colab import userdata

# Colab 비밀 저장소에서 ngrok 인증 토큰 가져오기
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

# ngrok 인증
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok가 설치되고 인증되었습니다.")

In [ ]:
# Ollama 서버에 대한 ngrok 터널 시작 (포트 11434)
# 터널을 이미 실행 중인 경우 닫고 새로 시작합니다.
ngrok.kill()

public_url = ngrok.connect(11434)
print(f"Ollama 서버가 다음 주소에서 공개적으로 액세스 가능합니다: {public_url}")

In [ ]:
import requests
import json
from pyngrok import ngrok

# Get the public URL from the ngrok tunnel
# Assuming the ngrok tunnel is still active from the previous step
try:
    # Get all active tunnels
    tunnels = ngrok.get_tunnels()
    public_url = None
    for tunnel in tunnels:
        if '11434' in tunnel.data['config']['addr']:
            public_url = tunnel.public_url
            break

    if not public_url:
        print("Error: No ngrok tunnel found for port 11434. Please re-run the ngrok connection cell.")
    else:
        ollama_api_url = f"{public_url}/api/generate"

        headers = {"Content-Type": "application/json"}
        payload = {
            "model": "qwen3.5",
            "prompt": "Explain why the sky is blue in one sentence.",
            "stream": False
        }

        print(f"Sending test request to Ollama at: {ollama_api_url}")
        response = requests.post(ollama_api_url, headers=headers, data=json.dumps(payload))

        if response.status_code == 200:
            result = response.json()
            print("\nOllama Response:")
            print(result['response'])
        else:
            print(f"Error: Request failed with status code {response.status_code}")
            print(response.text)
except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the ngrok tunnel is active and try again.")

PowerShell에서 다음 명령을 실행하여 Ollama 서버에 쿼리할 수 있습니다. `YOUR_NGROK_URL` 부분을 이전 단계에서 얻은 실제 ngrok URL로 교체해야 합니다.

In [ ]:
print("```powershell\n$ngrokUrl = \"https://prorate-mumbo-duty.ngrok-free.dev\" # 여기에 실제 ngrok URL을 입력하세요.\n$apiUrl = \"$($ngrokUrl)/api/generate\"\n\n$headers = @{\n    \"Content-Type\" = \"application/json\"\n}\n\n$body = @{\n    model = \"mistral\"\n    prompt = \"Hello Ollama, how are you?\"\n    stream = $false\n} | ConvertTo-Json\n\nInvoke-RestMethod -Uri $apiUrl -Method Post -Headers $headers -Body $body\n```")

위 명령은 `qwen3.5` 모델에 'Hello Ollama, how are you?'라는 프롬프트로 요청을 보냅니다. `prompt` 값을 변경하여 다른 질문을 할 수 있습니다.